### 3.1 (b)

In [101]:
import gurobipy as gp
from gurobipy import GRB

S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]
# p = [0.15, 0.30, 0.30, 0.20, 0.05]   # 교재 시나리오 값

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

model = gp.Model("RiskNeutralSLP_MaxProfit")

x = model.addVars(4, lb=0, name="x")
y = model.addVars(3, S, lb=0, name="y")

model.setObjective(
    gp.quicksum(
        p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3))
        for s in S
    ) - gp.quicksum(cost_x[j] * x[j] for j in range(4)),
    GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s] <= d_a[s])
    model.addConstr(y[1, s] <= d_b[s])
    model.addConstr(y[2, s] <= d_c[s])

# Solve
model.optimize()

# Print solution
if model.status == GRB.OPTIMAL:
    print("[Risk-Neutral SLP with Recourse]")
    print("Objective Value:", model.objVal)

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 40 rows, 19 columns and 102 nonzeros
Model fingerprint: 0x59f1cc23
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 21 rows, 19 columns, 83 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.6988095e+03   4.092262e+01   0.000000e+00      0s
       9    2.3410000e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.341000000e+03
[Risk-Neutral SLP with Recourse]
Objective Value: 2340.999999999998


In [102]:
print("\n[Risk-Neutral SLP with Recourse]")
print(f"z_{{RP}}^* = {model.objVal:.2f}")
print()

# First-stage solution
print("First-stage decision variables:")
for j in range(4):
    print(f"  x_{j+1} = {x[j].X:.2f}")
print()

# Second-stage solution
print("Second-stage decisions (y_i^s):")
for s in S:
    print(f"  Scenario s = {s+1}")
    for i in range(3):
        val = y[i, s].X
        if abs(val) > 1e-4:
            print(f"    y_{i+1}^{s+1} = {val:.2f}")
    print()


[Risk-Neutral SLP with Recourse]
z_{RP}^* = 2341.00

First-stage decision variables:
  x_1 = 236.40
  x_2 = 690.00
  x_3 = 432.00
  x_4 = 318.00

Second-stage decisions (y_i^s):
  Scenario s = 1
    y_1^1 = 15.00
    y_2^1 = 10.00
    y_3^1 = 5.00

  Scenario s = 2
    y_2^2 = 10.80
    y_3^2 = 15.00

  Scenario s = 3
    y_2^3 = 10.80
    y_3^3 = 15.00

  Scenario s = 4
    y_2^4 = 10.80
    y_3^4 = 15.00

  Scenario s = 5
    y_1^5 = 8.00
    y_2^5 = 10.00
    y_3^5 = 10.00



### 3.2 (a)

In [103]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("EV_Profit_Maximization")

x = model.addVars(4, lb=0, name="x")  # x_1 ~ x_4
y = model.addVars(3, lb=0, name="y")  # y_1 ~ y_3

model.setObjective(
    (1150 * y[0] + 1525 * y[1] + 1900 * y[2]) - (50 * x[0] + 30 * x[1] + 15 * x[2] + 10 * x[3]),
    GRB.MAXIMIZE
)

model.addConstr(x[0] <= 300)
model.addConstr(x[1] <= 700)
model.addConstr(x[2] <= 600)
model.addConstr(x[3] <= 500)
model.addConstr(x[1] + x[2] + x[3] <= 1600)

model.addConstr(x[0] >= 6 * y[0] + 8 * y[1] + 10 * y[2])
model.addConstr(x[1] >= 20 * y[0] + 25 * y[1] + 28 * y[2])
model.addConstr(x[2] >= 12 * y[0] + 15 * y[1] + 18 * y[2])
model.addConstr(x[3] >= 8 * y[0] + 10 * y[1] + 14 * y[2])

model.addConstr(y[0] <= 22.0)
model.addConstr(y[1] <= 17.5)
model.addConstr(y[2] <= 19.5)

# model.addConstr(y[0] <= 22.5)
# model.addConstr(y[1] <= 17.5)
# model.addConstr(y[2] <= 19.25)

model.optimize()

print("\n[EV Solution]")
print(f"z_EV^* = {model.objVal:.2f}")
for i in range(4):
    print(f"x_{i+1} = {x[i].X:.2f}")
for i in range(3):
    print(f"y_{i+1} = {y[i].X:.2f}")

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 12 rows, 7 columns and 26 nonzeros
Model fingerprint: 0xae124ecf
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+01, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 2e+03]
Presolve removed 8 rows and 2 columns
Presolve time: 0.00s
Presolved: 4 rows, 5 columns, 13 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.9257500e+04   1.416875e+02   0.000000e+00      0s
       3    3.2330000e+03   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.00 seconds (0.00 work units)
Optimal objective  3.233000000e+03

[EV Solution]
z_EV^* = 3233.00
x_1 = 244.28
x_2 = 700.00
x_3 = 443.40
x_4 = 334.60
y_1 = 0.00
y_2 = 6.16
y_3 = 19.50


In [104]:
S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]
# p = [0.15, 0.30, 0.30, 0.20, 0.05]   # 교재 시나리오 값

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

model = gp.Model("z_eev")

x = [244.28, 700, 443.4, 334.6] #z_EV solution
y = model.addVars(3, S, lb=0, name="y")

model.setObjective(
    gp.quicksum(
        p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3))
        for s in S
    ) - gp.quicksum(cost_x[j] * x[j] for j in range(4)),
    GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s] <= d_a[s])
    model.addConstr(y[1, s] <= d_b[s])
    model.addConstr(y[2, s] <= d_c[s])

# Solve
model.optimize()

# Print solution
if model.status == GRB.OPTIMAL:
    print("[Risk-Neutral SLP with Recourse]")
    print("z_EEV:", model.objVal)


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 40 rows, 15 columns and 75 nonzeros
Model fingerprint: 0x35b19bc3
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+02, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 7e+02]
Presolve removed 26 rows and 3 columns
Presolve time: 0.00s
Presolved: 14 rows, 12 columns, 42 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.1952667e+03   7.729250e+01   0.000000e+00      0s
      11    2.2875000e+03   0.000000e+00   0.000000e+00      0s

Solved in 11 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.287500000e+03
[Risk-Neutral SLP with Recourse]
z_EEV: 2287.5


### 3.2 (b)

In [105]:
# Create model
m = gp.Model("EES")

# First-stage variables
x1 = m.addVar(name="x1", lb=0)
x2 = m.addVar(name="x2", lb=0)
x3 = m.addVar(name="x3", lb=0)
x4 = m.addVar(name="x4", lb=0)

# Second-stage variables
y1 = m.addVar(name="y1", lb=0)
y2 = m.addVar(name="y2", lb=0)
y3 = m.addVar(name="y3", lb=0)

# Objective
m.setObjective(
    (1150 * y1 + 1525 * y2 + 1900 * y3) - (50 * x1 + 30 * x2 + 15 * x3 + 10 * x4),
    GRB.MAXIMIZE
)

# Constraints
m.addConstr(x1 <= 300)
m.addConstr(x2 <= 700)
m.addConstr(x3 <= 600)
m.addConstr(x4 <= 500)
m.addConstr(x2 + x3 + x4 <= 1600)

m.addConstr(x1 - 6 * y1 - 8 * y2 - 10 * y3 >= 0)
m.addConstr(x2 - 20 * y1 - 25 * y2 - 28 * y3 >= 0)
m.addConstr(x3 - 12 * y1 - 15 * y2 - 18 * y3 >= 0)
m.addConstr(x4 - 8 * y1 - 10 * y2 - 14 * y3 >= 0)

# Extreme event constraints
for b in [15, 20, 25, 30, 10]:
    m.addConstr(y1 <= b)
for b in [10, 15, 20, 25, 10]:
    m.addConstr(y2 <= b)
for b in [5, 15, 25, 30, 10]:
    m.addConstr(y3 <= b)

# Optimize
m.optimize()

# Print solution
if m.status == GRB.OPTIMAL:
    print(f"z_EES^* = {m.objVal:.2f}")
    print(f"x_1 = {x1.X:.2f}")
    print(f"x_2 = {x2.X:.2f}")
    print(f"x_3 = {x3.X:.2f}")
    print(f"x_4 = {x4.X:.2f}")
    print(f"y_1 = {y1.X:.2f}")
    print(f"y_2 = {y2.X:.2f}")
    print(f"y_3 = {y3.X:.2f}")

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 24 rows, 7 columns and 38 nonzeros
Model fingerprint: 0xc16a6d16
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+01, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 24 rows and 7 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.2500000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.250000000e+03
z_EES^* = 1250.00
x_1 = 130.00
x_2 = 390.00
x_3 = 240.00
x_4 = 170.00
y_1 = 0.00
y_2 = 10.00
y_3 = 5.00


In [106]:
S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

model = gp.Model("z_eees")

x = [130, 390, 240, 170] #z_ees solution
y = model.addVars(3, S, lb=0, name="y")

model.setObjective(
    gp.quicksum(
        p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3))
        for s in S
    ) - gp.quicksum(cost_x[j] * x[j] for j in range(4)),
    GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s] <= d_a[s])
    model.addConstr(y[1, s] <= d_b[s])
    model.addConstr(y[2, s] <= d_c[s])

# Solve
model.optimize()

# Print solution
if model.status == GRB.OPTIMAL:
    print("[Risk-Neutral SLP with Recourse]")
    print("E[z_EES]:", model.objVal)

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 40 rows, 15 columns and 75 nonzeros
Model fingerprint: 0xd6ece0f4
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+02, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 4e+02]
Presolve removed 20 rows and 0 columns
Presolve time: 0.00s
Presolved: 20 rows, 15 columns, 60 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.4166667e+03   4.664683e+01   0.000000e+00      0s
      14    1.2500000e+03   0.000000e+00   0.000000e+00      0s

Solved in 14 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.250000000e+03
[Risk-Neutral SLP with Recourse]
E[z_EES]: 1250.0


### 3.2(c)

In [107]:
from gurobipy import Model, GRB

# 시나리오별 수요와 확률
scenarios = {
    1: {"da": 15, "db": 10, "dc": 5, "p": 0.10},
    2: {"da": 20, "db": 15, "dc": 15, "p": 0.30},
    3: {"da": 25, "db": 20, "dc": 25, "p": 0.30},
    4: {"da": 30, "db": 25, "dc": 30, "p": 0.20},
    5: {"da": 10, "db": 10, "dc": 10, "p": 0.10},
}

results = {}

for s, data in scenarios.items():
    m = Model(f"Scenario_{s}")
    m.setParam("OutputFlag", 0)

    # Decision variables
    x1 = m.addVar(lb=0, name="x1")
    x2 = m.addVar(lb=0, name="x2")
    x3 = m.addVar(lb=0, name="x3")
    x4 = m.addVar(lb=0, name="x4")
    y1 = m.addVar(lb=0, name="y1")
    y2 = m.addVar(lb=0, name="y2")
    y3 = m.addVar(lb=0, name="y3")

    # Objective function
    m.setObjective(
        1150*y1 + 1525*y2 + 1900*y3 - (50*x1 + 30*x2 + 15*x3 + 10*x4),
        GRB.MAXIMIZE
    )

    # Constraints
    m.addConstr(x1 <= 300)
    m.addConstr(x2 <= 700)
    m.addConstr(x3 <= 600)
    m.addConstr(x4 <= 500)
    m.addConstr(x2 + x3 + x4 <= 1600)
    m.addConstr(6*y1 + 8*y2 + 10*y3 <= x1)
    m.addConstr(20*y1 + 25*y2 + 28*y3 <= x2)
    m.addConstr(12*y1 + 15*y2 + 18*y3 <= x3)
    m.addConstr(8*y1 + 10*y2 + 14*y3 <= x4)
    m.addConstr(y1 <= data["da"])
    m.addConstr(y2 <= data["db"])
    m.addConstr(y3 <= data["dc"])

    # Optimize
    m.optimize()

    results[s] = {
        "z_sa": m.objVal,
        "prob": data["p"],
        "x": [x1.X, x2.X, x3.X, x4.X],
        "y": [y1.X, y2.X, y3.X]
    }

# Expected value
z_sa_expected = sum(results[s]["z_sa"] * results[s]["prob"] for s in results)

print("[Scenario-wise Solutions]")
for s in results:
    print(f"Scenario {s} (p={results[s]['prob']}):")
    print(f"  z_sa = {results[s]['z_sa']:.2f}")
    print(f"  x = {results[s]['x']}")
    print(f"  y = {results[s]['y']}")
    print()

print(f"\nExpected value z_sa^* = {z_sa_expected:.2f}")

[Scenario-wise Solutions]
Scenario 1 (p=0.1):
  z_sa = 1250.00
  x = [130.0, 390.0, 240.0, 170.0]
  y = [0.0, 10.0, 5.0]

Scenario 2 (p=0.3):
  z_sa = 2810.00
  x = [239.6, 700.0, 438.0, 322.0]
  y = [0.0, 11.2, 15.0]

Scenario 3 (p=0.3):
  z_sa = 3750.00
  x = [250.0, 700.0, 450.0, 350.0]
  y = [0.0, 0.0, 25.0]

Scenario 4 (p=0.2):
  z_sa = 3750.00
  x = [250.0, 700.0, 450.00000000000006, 350.0]
  y = [0.0, 0.0, 25.0]

Scenario 5 (p=0.1):
  z_sa = 2000.00
  x = [180.0, 530.0, 330.0, 240.0]
  y = [0.0, 10.0, 10.0]


Expected value z_sa^* = 3043.00


In [108]:
S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

model = gp.Model("E[z_sa]")

# x = [130, 390, 240, 170] #z_sa_1
# x = [239.6, 700.0, 438.0, 322.0] #z_sa_2
# x = [250.0, 700.0, 450.0, 350.0] #z_sa_3
# x = [250.0, 700.0, 450.0, 350.0] #z_sa_4
x = [180.0, 530.0, 330.0, 240.0] #z_sa_5


y = model.addVars(3, S, lb=0, name="y")

model.setObjective(
    gp.quicksum(
        p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3))
        for s in S
    ) - gp.quicksum(cost_x[j] * x[j] for j in range(4)),
    GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s] <= d_a[s])
    model.addConstr(y[1, s] <= d_b[s])
    model.addConstr(y[2, s] <= d_c[s])

# Solve
model.optimize()

# Print solution
if model.status == GRB.OPTIMAL:
    print("[Risk-Neutral SLP with Recourse]")
    print("E[z_EES]:", model.objVal)

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 40 rows, 15 columns and 75 nonzeros
Model fingerprint: 0x7945189b
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+02, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 5e+02]
Presolve removed 20 rows and 0 columns
Presolve time: 0.00s
Presolved: 20 rows, 15 columns, 60 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.2500000e+03   6.909750e+01   0.000000e+00      0s
      15    1.8550000e+03   0.000000e+00   0.000000e+00      0s

Solved in 15 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.855000000e+03
[Risk-Neutral SLP with Recourse]
E[z_EES]: 1854.9999999999927


### 3.2(d)

In [109]:
2341.00 -3043

-702.0

In [110]:
2341.00 - 2287.50

53.5

### 3.3

In [111]:
import gurobipy as gp
from gurobipy import GRB

S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]
# p = [0.15, 0.30, 0.30, 0.20, 0.05]   # 교재 시나리오 값

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

M = 500000
lamb = 1000
# lamb=0
eta = 0
# eta = 2341

model = gp.Model("Excess_Probability")
model.setParam("MIPGap", 0.000001)

x = model.addVars(4, lb=0, name="x")
y = model.addVars(3, S, lb=0, name="y")
gamma = model.addVars(S, vtype=GRB.BINARY, name="gamma")

model.setObjective(
    gp.quicksum(
        p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3))
        for s in S
    ) - gp.quicksum(cost_x[j] * x[j] for j in range(4)) - lamb * gp.quicksum(p[s] * gamma[s] for s in S), GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s] <= d_a[s])
    model.addConstr(y[1, s] <= d_b[s])
    model.addConstr(y[2, s] <= d_c[s])
    
    model.addConstr(
        gp.quicksum(profit[i] * y[i, s] for i in range(3))
        - gp.quicksum(cost_x[j] * x[j] for j in range(4))
        + M * gamma[s]
        >= eta
    )

model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"Objective Value: {model.objVal:.2f}")
    x_vals = [x[i].X for i in range(4)]
    print("x:", x_vals)
    gamma_vals = [gamma[s].X for s in S]
    print("gamma:", gamma_vals)

    y_vals = {}
    for s in S:
        y_vals[s] = [y[i, s].X for i in range(3)]
        print(f"y (Scenario {s+1}):", y_vals[s])

    expected_profit = sum(
        p[s] * sum(profit[i] * y_vals[s][i] for i in range(3)) for s in S
    ) - sum(cost_x[j] * x_vals[j] for j in range(4))

    print(f"Expected Profit (excluding risk term): {expected_profit:.2f}")

Set parameter MIPGap to value 1e-06
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-06

Optimize a model with 45 rows, 24 columns and 142 nonzeros
Model fingerprint: 0x62d70392
Variable types: 19 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+05]
  Objective range  [1e+01, 6e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 2e+03]
Found heuristic solution: objective -0.0000000
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 26 rows, 24 columns, 123 nonzeros
Variable types: 19 continuous, 5 integer (5 binary)

Root relaxation: objective 2.340640e+03, 10 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

### 3.4

In [9]:
import gurobipy as gp
from gurobipy import GRB

S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]
# p = [0.15, 0.30, 0.30, 0.20, 0.05]   # 교재 시나리오 값

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

model = gp.Model("CVaR")
model.setParam("MIPGap", 0.000001)

x = model.addVars(4, lb=0, name="x")
y = model.addVars(3, S, lb=0, name="y")
lamb = 100
eta = model.addVar(lb=-GRB.INFINITY, name="VaR")
nu = model.addVars(S, lb=0, name="nu")
alpha = 0.95


model.setObjective(
    gp.quicksum(
        p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3))
        for s in S
    )
    - gp.quicksum(cost_x[j] * x[j] for j in range(4))
    - lamb * eta
    - lamb / (1 - alpha) * gp.quicksum(p[s] * nu[s] for s in S),
    GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s] <= d_a[s])
    model.addConstr(y[1, s] <= d_b[s])
    model.addConstr(y[2, s] <= d_c[s])
    
    model.addConstr(-gp.quicksum(cost_x[j] * x[j] for j in range(4))+gp.quicksum(profit[i] * y[i, s] for i in range(3))+eta+nu[s]>=0)

model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"Objective Value: {model.objVal:.2f}")

    x_vals = [x[i].X for i in range(4)]
    print("x:", x_vals)

    print(f"eta: {eta.X:.2f}")

    y_vals = {}
    nu_vals = {} 
    for s in S:
        y_vals[s] = [y[i, s].X for i in range(3)]
        nu_vals[s] = nu[s].X
        print(f"y (Scenario {s+1}):", y_vals[s])
        print(f"nu (Scenario {s+1}):", nu_vals[s])

    expected_profit = sum(
        p[s] * sum(profit[i] * y_vals[s][i] for i in range(3)) for s in S
    ) - sum(cost_x[j] * x_vals[j] for j in range(4))

    print(f"Expected Profit (excluding risk term): {expected_profit:.2f}")

Set parameter MIPGap to value 1e-06
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-06

Optimize a model with 45 rows, 25 columns and 147 nonzeros
Model fingerprint: 0x9fd9569a
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve time: 0.00s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Infeasible or unbounded model


### 3.5

In [5]:
import gurobipy as gp
from gurobipy import GRB

S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]
# p = [0.15, 0.30, 0.30, 0.20, 0.05]   # 교재 시나리오 값

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

model = gp.Model("EE")
model.setParam("MIPGap", 0.000001)

x = model.addVars(4, lb=0, name="x")
y = model.addVars(3, S, lb=0, name="y")
lamb = 1000
eta = 2341
nu = model.addVars(S, lb=0, name="nu")


model.setObjective(
    gp.quicksum(
        p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3))
        for s in S
    ) - gp.quicksum(cost_x[j] * x[j] for j in range(4)) - lamb * gp.quicksum(p[s] * nu[s] for s in S), GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s] <= d_a[s])
    model.addConstr(y[1, s] <= d_b[s])
    model.addConstr(y[2, s] <= d_c[s])
    
    model.addConstr(-gp.quicksum(cost_x[j] * x[j] for j in range(4))+gp.quicksum(profit[i] * y[i, s] for i in range(3))+nu[s]>=-eta)

model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"Objective Value: {model.objVal:.2f}")

    x_vals = [x[i].X for i in range(4)]
    print("x:", x_vals)

    y_vals = {}
    nu_vals = {} 
    for s in S:
        y_vals[s] = [y[i, s].X for i in range(3)]
        nu_vals[s] = nu[s].X
        print(f"y (Scenario {s+1}):", y_vals[s])
        print(f"nu (Scenario {s+1}):", nu_vals[s])

    expected_profit = sum(
        p[s] * sum(profit[i] * y_vals[s][i] for i in range(3)) for s in S
    ) - sum(cost_x[j] * x_vals[j] for j in range(4))

    print(f"Expected Profit (excluding risk term): {expected_profit:.2f}")

Set parameter MIPGap to value 1e-06
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-06

Optimize a model with 45 rows, 24 columns and 142 nonzeros
Model fingerprint: 0xdb6c54f4
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 26 rows, 24 columns, 123 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.6988095e+03   1.255952e+03   0.000000e+00      0s
       9    2.3410000e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.341000000e+03
Objective Value: 2341.00
x: [236.4, 690.0, 432.0, 318.0]
y (Scenario 1): [15.0, 10.0, 5.0]
nu (Scenari

### 3.6

In [2]:
import gurobipy as gp
from gurobipy import GRB

S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]
# p = [0.15, 0.30, 0.30, 0.20, 0.05]   # 교재 시나리오 값

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

model = gp.Model("pc-sp")
model.setParam("MIPGap", 0.000001)

x = model.addVars(4, lb=0, name="x")
y = model.addVars(3, S, lb=0, name="y")
z = model.addVars(S, lb=0, vtype = GRB.BINARY, name="z")
M=500000
alpha = 0.8


model.setObjective(
    gp.quicksum(
        p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3))
        for s in S
    )
    - gp.quicksum(cost_x[j] * x[j] for j in range(4)),
    GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s] - M*z[s]<= d_a[s])
    model.addConstr(y[1, s] - M*z[s]<= d_b[s])
    model.addConstr(y[2, s] - M*z[s]<= d_c[s])
    
model.addConstr(gp.quicksum(p[s] * z[s] for s in S) <= 1 - alpha)

model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"Objective Value: {model.objVal:.2f}")

    x_vals = [x[i].X for i in range(4)]
    print("x:", x_vals)

    y_vals = {}
    z_vals = {} 
    for s in S:
        y_vals[s] = [y[i, s].X for i in range(3)]
        z_vals[s] = z[s].X
        print(f"y (Scenario {s+1}):", y_vals[s])
        print(f"z (Scenario {s+1}):", z_vals[s])

    expected_profit = sum(
        p[s] * sum(profit[i] * y_vals[s][i] for i in range(3)) for s in S
    ) - sum(cost_x[j] * x_vals[j] for j in range(4))

    print(f"Expected Profit (excluding risk term): {expected_profit:.2f}")

Set parameter MIPGap to value 1e-06
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-06

Optimize a model with 41 rows, 24 columns and 122 nonzeros
Model fingerprint: 0x812e5cb7
Variable types: 19 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e-01, 5e+05]
  Objective range  [1e+01, 6e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-01, 2e+03]
Found heuristic solution: objective -0.0000000
Presolve removed 12 rows and 4 columns
Presolve time: 0.00s
Presolved: 29 rows, 20 columns, 99 nonzeros
Variable types: 19 continuous, 1 integer (1 binary)

Root relaxation: objective 3.174000e+03, 13 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time


### 3.7

In [69]:
import gurobipy as gp
from gurobipy import GRB

S = range(5) 
p = [0.10, 0.30, 0.30, 0.20, 0.10]
# p = [0.15, 0.30, 0.30, 0.20, 0.05]   # 교재 시나리오 값

d_a = [15, 20, 25, 30, 10]
d_b = [10, 15, 20, 25, 10]
d_c = [5, 15, 25, 30, 10]

profit = [1150, 1525, 1900]
cost_x = [50, 30, 15, 10]

W = [
    [6, 8, 10],
    [20, 25, 28],
    [12, 15, 18],
    [8, 10, 14]
]

x_bounds = [300, 700, 600, 500]
total_time_limit = 1600

model = gp.Model("qdev")
model.setParam("MIPGap", 0.000001)

x = model.addVars(4, lb=0, name="x")
y = model.addVars(3, S, lb=0, name="y")
u = model.addVars(S, lb=0, name="u")
v = model.addVars(S, lb=0, name="v")
eta = model.addVars(1, lb=-GRB.INFINITY, name = "eta")
ep1 = 0.1
ep2 =0.19
lamb = 1/ep2


qdev_risk = lamb * (ep1 * gp.quicksum(p[s] * u[s] for s in S) + ep2 * gp.quicksum(p[s] * v[s] for s in S))

model.setObjective(
    gp.quicksum(p[s] * gp.quicksum(profit[i] * y[i, s] for i in range(3)) for s in S)
    - gp.quicksum(cost_x[j] * x[j] for j in range(4))
    - qdev_risk,
    GRB.MAXIMIZE
)

for j in range(4):
    model.addConstr(x[j] <= x_bounds[j])
model.addConstr(x[1] + x[2] + x[3] <= total_time_limit)

for s in S:
    model.addConstr(sum(W[0][i] * y[i, s] for i in range(3)) <= x[0])
    model.addConstr(sum(W[1][i] * y[i, s] for i in range(3)) <= x[1])
    model.addConstr(sum(W[2][i] * y[i, s] for i in range(3)) <= x[2])
    model.addConstr(sum(W[3][i] * y[i, s] for i in range(3)) <= x[3])

    model.addConstr(y[0, s]<= d_a[s])
    model.addConstr(y[1, s]<= d_b[s])
    model.addConstr(y[2, s]<= d_c[s])
    
for s in S:
    profit_expr = gp.quicksum(profit[i] * y[i, s] for i in range(3))
    cost_expr = gp.quicksum(cost_x[j] * x[j] for j in range(4))

    model.addConstr(u[s] + profit_expr - cost_expr - eta[0] >= 0)
    model.addConstr(v[s] - profit_expr + cost_expr + eta[0] >= 0)

model.optimize()

if model.status == GRB.OPTIMAL:
    print(lamb)
    print(f"Objective Value (including risk penalty): {model.objVal:.2f}")

    x_vals = [x[i].X for i in range(4)]
    print("\nx (First-stage decisions):")
    for i in range(4):
        print(f"x[{i+1}] = {x_vals[i]:.1f}")

    y_vals = {}
    u_vals = {}
    v_vals = {}

    print("\ny (Second-stage decisions per scenario):")
    for s in S:
        y_vals[s] = [y[i, s].X for i in range(3)]
        u_vals[s] = u[s].X
        v_vals[s] = v[s].X
        print(f"Scenario {s+1}: y = {y_vals[s]}, u = {u_vals[s]:.2f}, v = {v_vals[s]:.2f}")

    eta_val = eta[0].X
    print(f"\nEta (α-quantile profit): η = {eta_val:.2f}")

    expected_profit = sum(
        p[s] * sum(profit[i] * y_vals[s][i] for i in range(3)) for s in S
    ) - sum(cost_x[j] * x_vals[j] for j in range(4))

    qdev_penalty = lamb * (
        ep1 * sum(p[s] * u_vals[s] for s in S) +
        ep2 * sum(p[s] * v_vals[s] for s in S)
    )

    print(f"\nExpected Profit (excluding risk term): {expected_profit:.2f}")
    print(f"QDEV Penalty Term: {qdev_penalty:.2f}")
    print(f"Expected Profit - QDEV (Objective): {expected_profit - qdev_penalty:.2f}")

Set parameter MIPGap to value 1e-06
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-06

Optimize a model with 50 rows, 30 columns and 192 nonzeros
Model fingerprint: 0x9202bc03
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [5e-02, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 31 rows, 30 columns, 173 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
      18    2.1046842e+03   0.000000e+00   0.000000e+00      0s

Solved in 18 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.104684211e+03
5.2631578947368425
Objective Value (including risk penalty): 2104.68

x (First-stage decisions):
x[1]